# Project 4: multi-hop agent vs single-shot RAG on HotpotQA

Runs three modes with the same model (Qwen2.5-7B-Instruct, 4-bit) on the same 100 questions:

| mode | what it does |
|---|---|
| closed-book | answers from memory, no search |
| single-shot RAG | one search with the raw question, then answers from the top 5 passages |
| agent | the `agentlab` loop: the model chooses searches, reads results, then finishes |

**Before you run:** switch the runtime to a T4 GPU (Runtime > Change runtime type), and push the latest `04-agents` to GitHub, because cell P4-1 installs `agentlab` from your repo.

**Order:** P4-1 and P4-2 are quick. P4-3 downloads about 15 GB of model weights the first time, then runs a 3-question smoke test: read that output before starting P4-4. P4-4 is the long run. It saves every result as it goes and skips finished questions if you run it again, so a disconnect does not lose work.

### P4-1: install

In [ ]:
# Installs agentlab from your GitHub repo. --force-reinstall makes sure you get the latest push.
!pip install -q --no-deps --force-reinstall "git+https://github.com/kousthub93/llm-systems-lab.git#subdirectory=04-agents"
!pip install -q sentence-transformers rank_bm25 accelerate bitsandbytes datasets

import agentlab
from agentlab.hotpot import build_hotpot      # fails here, early, if the HotpotQA step was not pushed yet
print("agentlab loaded from", agentlab.__file__)

### P4-2: config, data, retriever

Loads HotpotQA's distractor dev set, pools the paragraphs of `N_CORPUS` questions into one searchable corpus, and builds the Project 2 hybrid retriever (bge-small + BM25 + RRF, MiniLM re-rank of 20). The last lines print the **retrieval ceiling**: how often the raw question, searched once, brings back both supporting paragraphs. Single-shot RAG cannot beat that number, and the agent is trying to.

In [ ]:
import json, os, time, urllib.request
import torch

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"     # Apache 2.0
N_EVAL, N_CORPUS, SEED = 100, 1000, 0     # evaluate on 100 questions; index the paragraphs of 1000
SEARCH_K, SINGLE_SHOT_K = 3, 5            # passages per agent search / in the single-shot prompt
MAX_STEPS = 6

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run again."

def load_from_hf():
    """Read the distractor/validation parquet file(s) straight from the Hub's parquet branch."""
    import pandas as pd
    from huggingface_hub import HfApi, hf_hub_download
    repo, rev = "hotpotqa/hotpot_qa", "refs/convert/parquet"
    files = HfApi().list_repo_files(repo, repo_type="dataset", revision=rev)
    wanted = sorted(f for f in files if f.endswith(".parquet") and "distractor" in f and "validation" in f)
    if not wanted:
        raise RuntimeError("no distractor validation parquet on that branch. Files there: " + ", ".join(files[:40]))
    print("reading", wanted)
    frames = [pd.read_parquet(hf_hub_download(repo, f, repo_type="dataset", revision=rev)) for f in wanted]
    return pd.concat(frames, ignore_index=True).to_dict("records")


def load_from_cmu():
    """The original release. Its host did not resolve from Colab on 2026-11-15, so this is only a fallback."""
    url, path = "http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_dev_distractor_v1.json", "hotpot_dev_distractor_v1.json"
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)               # never leave a half-downloaded file behind
    return json.load(open(path))


rows = None
for name, loader in (("Hugging Face parquet", load_from_hf), ("CMU JSON", load_from_cmu)):
    try:
        rows = loader()
        print("loaded from", name)
        break
    except Exception as e:
        print(f"{name} failed: {e!r}")
if rows is None:
    raise RuntimeError("Could not load HotpotQA from either source. Paste the messages above and we will fix the loader.")
print(len(rows), "rows loaded")

from agentlab.hotpot import build_hotpot, title_recall
from agentlab.retrieval import HybridRetriever, make_search_tool

examples, passages, stats = build_hotpot(rows, N_EVAL, N_CORPUS, seed=SEED)
print(stats)
ex = examples[0]
print("\nexample:", ex["question"], "| answer:", ex["answer"], "| type:", ex["type"], "| supporting:", ex["gold_titles"])

t0 = time.time()
retriever = HybridRetriever.from_models(passages, pool=20)
print(f"\nindexed {len(passages)} passages in {time.time() - t0:.0f}s")

print("\nretrieval ceiling (raw question, searched once):")
for k in (3, 5, 10):
    print(" ", title_recall(retriever, examples, k))

### P4-3: load Qwen, build the agent, smoke test

Prints the system prompt, then runs 3 questions through the agent and shows every step: what the model said, how the loop parsed it, and what came back. **Read these traces before running P4-4.** Look for: valid JSON on the first try, sensible second queries, and `finish` with a short answer.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODEL_ID)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
model.eval()


class QwenChat:
    """messages -> reply text, greedy. Counts prompt tokens so runs can report them."""

    def __init__(self, model, tok, max_new_tokens=128):
        self.model, self.tok, self.max_new_tokens = model, tok, max_new_tokens
        self.tokens_in = self.tokens_out = 0

    def __call__(self, messages):
        text = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tok(text, return_tensors="pt").to(self.model.device)
        n_in = inputs["input_ids"].shape[1]
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens, do_sample=False,
                                      repetition_penalty=1.0, pad_token_id=self.tok.eos_token_id)
        new = out[0, n_in:]
        self.tokens_in += n_in
        self.tokens_out += len(new)
        return self.tok.decode(new, skip_special_tokens=True)


chat = QwenChat(model, tok)
chat([{"role": "user", "content": "Say hi."}])          # warm-up, so GPU start-up is not in any timing

from agentlab import Agent, ToolRegistry, build_system_prompt
from agentlab.runner import (HOTPOT_EXTRA, format_trace, run_agent, run_closed_book, run_single_shot,
                             summarize, summarize_by_type)

tools = ToolRegistry([make_search_tool(retriever, k=SEARCH_K)])
agent = Agent(chat, tools, max_steps=MAX_STEPS, system_prompt=build_system_prompt(tools, HOTPOT_EXTRA))

print("=== system prompt ===")
print(agent.system_prompt)

print("\n=== smoke test: 3 questions through the agent ===")
for ex in examples[:3]:
    print("\n" + "-" * 100)
    print("Q:", ex["question"], "\ngold:", ex["answer"], "| type:", ex["type"], "| supporting:", ex["gold_titles"])
    print(format_trace(agent.run(ex["question"]), width=400))

### P4-4: full run

Runs the three modes over all `N_EVAL` questions and appends each result to `project4_runs.jsonl` as it finishes. To run one mode at a time, edit `MODES`. Re-running skips questions already in the file. The agent mode is the slow one (several model calls per question, and a 4-bit 7B model on a T4 is not fast), so plan on this taking a while. Both output files download automatically at the end.

In [ ]:
import pandas as pd
from google.colab import files

OUT = "project4_runs.jsonl"
MODES = ["closed-book", "single-shot RAG", "agent"]

done = set()
if os.path.exists(OUT):
    for line in open(OUT):
        r = json.loads(line)
        done.add((r["mode"], r["id"]))
    print(f"resuming: {len(done)} results already in {OUT}")


def run_one(mode, ex):
    t0, tok0 = time.perf_counter(), chat.tokens_in
    if mode == "closed-book":
        rec = run_closed_book(chat, ex)
    elif mode == "single-shot RAG":
        rec = run_single_shot(chat, retriever, ex, k=SINGLE_SHOT_K)
    else:
        rec = run_agent(agent, ex)
    rec["wall_s"] = round(time.perf_counter() - t0, 2)          # includes retrieval time
    rec["prompt_tokens"] = chat.tokens_in - tok0                 # for the agent: summed over every model call
    return rec


t_start = time.time()
with open(OUT, "a") as f:
    for mode in MODES:
        todo = [ex for ex in examples if (mode, ex["id"]) not in done]
        for i, ex in enumerate(todo, 1):
            f.write(json.dumps(run_one(mode, ex)) + "\n")
            f.flush()
            if i % 10 == 0 or i == len(todo):
                print(f"{mode}: {i}/{len(todo)}  ({(time.time() - t_start) / 60:.1f} min elapsed)")

records = [json.loads(line) for line in open(OUT)]
summary = pd.DataFrame(summarize(records)).T
summary["n"] = summary["n"].astype(int)
by_type = pd.DataFrame(summarize_by_type(records)).T
by_type["n"] = by_type["n"].astype(int)
print("\n", summary.to_string())
print("\n", by_type.to_string())
summary.to_csv("project4_summary.csv")
files.download(OUT)
files.download("project4_summary.csv")